# SHAP Interpretation

This notebook provides a detailed interpretation of the final XGBoost model using SHAP (SHapley Additive exPlanations).

The purpose of this analysis is to understand how individual features influence the model's predictions and whether their values contribute to higher or lower predicted probabilities of future air quality deterioration.

Unlike built-in feature importance, SHAP analysis provides both the magnitude and direction of feature contributions, allowing the model's predictions to be interpreted at both the global and individual levels.

The analysis uses the already trained final XGBoost model without any additional training or hyperparameter tuning.


## 1. Load Final Model

The final XGBoost model trained on the combined 2013–2015 training and validation data is loaded from the saved model file.

The fitted model will be used directly for SHAP analysis without any additional training or hyperparameter tuning.


In [1]:
from pathlib import Path
import joblib
import pandas as pd
import shap


In [2]:
MODEL_PATH = Path("../models/xgboost_final.joblib")

final_xgb = joblib.load(MODEL_PATH)

print("Final XGBoost model loaded successfully.")
print(f"Model path: {MODEL_PATH}")

Final XGBoost model loaded successfully.
Model path: ..\models\xgboost_final.joblib


### Model Fitting Verification

The loaded XGBoost model is checked to confirm that the estimator was fitted successfully before proceeding with SHAP analysis.


In [3]:
is_fitted = hasattr(
    final_xgb.named_steps["model"],
    "n_features_in_"
)

print(f"XGBoost fitted: {is_fitted}")

XGBoost fitted: True


## 2. Prepare Test Data

The SHAP analysis is performed on the unseen test set used for the final model evaluation.

Using the test set ensures that the interpretation reflects model behavior on data that was not used during model development, hyperparameter tuning, or final training.

The test features are prepared using the same preprocessing pipeline included in the final XGBoost model.


In [4]:
DATA_PATH = Path("../data/processed/airshift_labeled.csv")

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["datetime"]
)

TEST_START = "2016-01-01"
TEST_END = "2017-02-28 17:00:00"

test_data = df[
    (df["datetime"] >= TEST_START) &
    (df["datetime"] <= TEST_END)
].copy()

print("Test data shape:", test_data.shape)
print("Test date range:")
print(test_data["datetime"].min())
print(test_data["datetime"].max())

Test data shape: (121900, 100)
Test date range:
2016-01-01 00:00:00
2017-02-28 17:00:00


## 3. Define SHAP Features and Target

The target variable is separated from the input features before applying SHAP analysis.

The `Deterioration` column is used as the target, while `No` and `datetime` are excluded because they were not used as model features during training.

The remaining columns correspond to the input features used by the final XGBoost model.


In [5]:
TARGET = "Deterioration"

EXCLUDE_COLUMNS = [
    TARGET,
    "No",
    "datetime"
]

X_test = test_data.drop(
    columns=EXCLUDE_COLUMNS
)

y_test = test_data[TARGET]

print("Test features shape:", X_test.shape)
print("Test target shape:", y_test.shape)
print("Number of features:", X_test.shape[1])

Test features shape: (121900, 97)
Test target shape: (121900,)
Number of features: 97


## 4. Transform Test Features

The test features are transformed using the fitted preprocessing step from the final XGBoost pipeline.

This applies the same numerical and categorical transformations used during model training, ensuring that the transformed test data is consistent with the input representation expected by the XGBoost model.


In [6]:
preprocessor = final_xgb.named_steps["preprocessor"]
xgb_model = final_xgb.named_steps["model"]

X_test_transformed = preprocessor.transform(X_test)

print("Transformed test data shape:", X_test_transformed.shape)

Transformed test data shape: (121900, 123)


## 5. Initialize SHAP Explainer

A SHAP TreeExplainer is initialized using the fitted XGBoost model.

TreeExplainer is designed for tree-based machine learning models and provides an efficient way to calculate SHAP values for XGBoost predictions.

The explainer is created from the already trained model without modifying or retraining it.


In [7]:
explainer = shap.TreeExplainer(xgb_model)

print("SHAP TreeExplainer initialized successfully.")

SHAP TreeExplainer initialized successfully.


## 6. Select SHAP Sample

A representative sample of the unseen test set is selected for SHAP analysis to reduce computational cost while preserving reproducibility.

A fixed random seed is used so that the same observations can be selected if the analysis is repeated.

The sample is used for model interpretation only and does not affect the trained model or its test-set evaluation.


In [8]:
SHAP_SAMPLE_SIZE = 5000
RANDOM_STATE = 42

import numpy as np

rng = np.random.default_rng(RANDOM_STATE)

shap_indices = rng.choice(
    X_test_transformed.shape[0],
    size=SHAP_SAMPLE_SIZE,
    replace=False
)

X_shap = X_test_transformed[shap_indices]

print("SHAP sample shape:", X_shap.shape)

SHAP sample shape: (5000, 123)


## 7. Calculate SHAP Values

SHAP values are calculated for the selected test sample using the fitted XGBoost model and the initialized TreeExplainer.

Each SHAP value represents the contribution of a transformed feature to an individual model prediction. Positive and negative SHAP values indicate contributions in opposite directions relative to the model's expected prediction.

The resulting SHAP values will be used for global and local model interpretation.


In [9]:
shap_values = explainer.shap_values(X_shap)

print("SHAP values calculated successfully.")
print("SHAP values shape:", np.asarray(shap_values).shape)

SHAP values calculated successfully.
SHAP values shape: (5000, 123)
